# 🫁 Respiratory Disease Detection — CNNBiLSTMAttention v2
**Tessan x Snowflake Hackathon — Version optimisée**

### Améliorations vs v1
| | v1 | v2 |
|---|---|---|
| Entrée modèle | Mel seul (1 canal) | **Mel + MFCC (2 canaux)** |
| Epochs | 80 | **120 + patience 20** |
| Loss | CrossEntropy | **CrossEntropy + label_smoothing=0.1** |
| Scheduler | CosineAnnealing | **Warmup 5ep + CosineAnnealing** |
| Augmentation | SpecAugment agressif | **SpecAugment tunné + Mixup** |
| Inférence | 1 passe | **TTA (5 passes moyennées)** |

## 1. Imports & Session

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np
import os, tempfile, pickle

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, accuracy_score, confusion_matrix,
    classification_report, roc_auc_score
)
import matplotlib.pyplot as plt
import seaborn as sns

session = get_active_session()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Session active | Device: {DEVICE}")

## 2. Chargement des données

In [ ]:
TABLE_NAME = "M2_ISD_EQUIPE_1_DB.PROCESSED.TRAINING_DATA_V"
BASE_STAGE = "@M2_ISD_EQUIPE_1_DB.PROCESSED.STG_RESPIRATORY_FEATURES"
AUGMENTED_STAGE = "@M2_ISD_EQUIPE_1_DB.PROCESSED.STG_RESPIRATORY_FEATURES_AUG"

query = f"""
    SELECT CLASS, CLASS_WEIGHT, IS_AUGMENTED, FEAT_MEL, FEAT_MFCC
    FROM {TABLE_NAME}
"""
df = session.sql(query).to_pandas()

print(f"✅ {len(df)} échantillons chargés")
print("\n📊 Distribution des classes :")
print(df["CLASS"].value_counts())
print(f"\n🔁 Augmentés : {df['IS_AUGMENTED'].sum()} | Originaux : {(~df['IS_AUGMENTED']).sum()}")

## 3. Décodage des features — Mel + MFCC (2 canaux)

> **Nouveauté v2** : on charge les deux features et on les empile comme 2 canaux.
> Le modèle voit chaque sample comme une image à 2 canaux au lieu de 1.

In [ ]:
def decode_feature(file_name: str, class_name: str) -> np.ndarray:
    """Télécharge et charge un fichier .npy depuis l'un des deux Snowflake Stages."""
    
    # Liste des stages à parcourir dans l'ordre de priorité
    stages = [BASE_STAGE, AUGMENTED_STAGE]
    tmp_dir = tempfile.mkdtemp()
    
    for stage in stages:
        stage_path = f"{stage}/{class_name.lower()}/{file_name}"
        try:
            # On tente de récupérer le fichier
            session.file.get(stage_path, tmp_dir)
            local_file = os.path.join(tmp_dir, file_name)
            
            # Si le fichier existe localement après le GET, on le charge
            if os.path.exists(local_file):
                data = np.load(local_file)
                # Optionnel : nettoyer le fichier local immédiatement
                os.remove(local_file) 
                return data
        except Exception:
            # Si le fichier n'est pas dans ce stage, on passe au suivant sans erreur fatale
            continue

    # Si on arrive ici, c'est que le fichier n'a été trouvé dans aucun des deux stages
    print(f"⚠️ Fichier {file_name} introuvable dans les stages pour la classe {class_name}")
    return np.array([])


# ============================================================
# Chargement Mel + MFCC → tenseur 2 canaux (canal 0=Mel, canal 1=MFCC)
# ============================================================
print("⏳ Chargement des spectrogrammes Mel + MFCC...")
mel_tensors, labels = [], []
class_mapping = {c: i for i, c in enumerate(sorted(df["CLASS"].unique()))}
print(f"Classes : {class_mapping}")

skipped = 0
for idx, row in df.iterrows():
    # --- Mel ---
    mel = decode_feature(row["FEAT_MEL"], row["CLASS"])
    if mel.size == 0:
        skipped += 1
        continue
    mel_norm = (mel - mel.mean()) / (mel.std() + 1e-8)

    # --- MFCC ---
    mfcc = decode_feature(row["FEAT_MFCC"], row["CLASS"])
    if mfcc.size == 0:
        skipped += 1
        continue
    mfcc_norm = (mfcc - mfcc.mean()) / (mfcc.std() + 1e-8)

    # --- Redimensionner MFCC à la même shape que Mel (n_mels, T) ---
    mfcc_resized = F.interpolate(
        torch.tensor(mfcc_norm, dtype=torch.float32).unsqueeze(0).unsqueeze(0),
        size=(mel_norm.shape[0], mel_norm.shape[1]),
        mode="bilinear",
        align_corners=False
    ).squeeze(0).squeeze(0)

    # --- Stack 2 canaux : (2, n_mels, T) ---
    mel_t = torch.tensor(mel_norm, dtype=torch.float32)
    mel_tensors.append(torch.stack([mel_t, mfcc_resized], dim=0))
    labels.append(class_mapping[row["CLASS"]])

X = torch.stack(mel_tensors)          # (N, 2, n_mels, T)
y = torch.tensor(labels, dtype=torch.long)

print(f"\n✅ Shape X : {X.shape}  ← (N, 2 canaux, n_mels, time_frames)")
print(f"   Shape y : {y.shape}")
print(f"   Skipped : {skipped} samples")
print(f"   Canal 0 = Mel spectrogramme")
print(f"   Canal 1 = MFCC (redimensionné)")

## 4. Train / Val / Test split stratifié (70 / 15 / 15)

In [ ]:
indices = np.arange(len(y))
y_np    = y.numpy()

# 70% train — 30% temp
idx_train, idx_temp = train_test_split(
    indices, test_size=0.30, random_state=42, stratify=y_np
)
# 50/50 sur le reste → 15% val / 15% test
idx_val, idx_test = train_test_split(
    idx_temp, test_size=0.50, random_state=42, stratify=y_np[idx_temp]
)

X_train, y_train = X[idx_train], y[idx_train]
X_val,   y_val   = X[idx_val],   y[idx_val]
X_test,  y_test  = X[idx_test],  y[idx_test]

print(f"Train  : {len(X_train)} samples")
print(f"Val    : {len(X_val)} samples")
print(f"Test   : {len(X_test)} samples")

for split_name, split_y in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    vals, cnts = np.unique(split_y.numpy(), return_counts=True)
    dist = " | ".join(
        f"{list(class_mapping.keys())[list(class_mapping.values()).index(v)]}={c}"
        for v, c in zip(vals, cnts)
    )
    print(f"  {split_name}: {dist}")

## 5. Dataset PyTorch — SpecAugment tunné + Mixup

> **v2** : `freq_mask=10`, `time_mask=20` (moins agressif qu'avant sur ~1200 samples).
> Mixup ajouté comme méthode séparée, appelée dans la boucle d'entraînement.

In [ ]:
class MelDataset(Dataset):
    """
    Dataset PyTorch avec SpecAugment optionnel.
    Fonctionne avec N canaux (1 ou 2) sans modification.
    """

    def __init__(self, X, y, augment=False, freq_mask=10, time_mask=20):
        self.X = X
        self.y = y
        self.augment   = augment
        self.freq_mask = freq_mask   # v2 : réduit de 20→10
        self.time_mask = time_mask   # v2 : réduit de 30→20

    def __len__(self):
        return len(self.y)

    def _spec_augment(self, x):
        """Masquage aléatoire fréquence + temps. S'applique aux 2 canaux d'un coup."""
        x = x.clone()
        _, n_mels, n_frames = x.shape

        # Masquage fréquence
        f  = np.random.randint(0, max(1, self.freq_mask))
        f0 = np.random.randint(0, max(1, n_mels - f))
        x[:, f0:f0 + f, :] = 0

        # Masquage temps
        t  = np.random.randint(0, max(1, self.time_mask))
        t0 = np.random.randint(0, max(1, n_frames - t))
        x[:, :, t0:t0 + t] = 0

        return x

    def __getitem__(self, idx):
        x = self.X[idx]
        if self.augment:
            x = self._spec_augment(x)
        return x, self.y[idx]


def mixup_batch(x, y, alpha=0.2):
    """
    Mixup augmentation : mélange deux samples avec un poids lambda.
    Retourne x_mix, y_a, y_b, lambda pour calculer la loss combinée.
    """
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0))
    x_mix = lam * x + (1 - lam) * x[idx]
    return x_mix, y, y[idx], lam


# === WeightedRandomSampler — rééquilibrage au sampling ===
class_counts_train  = np.bincount(y_train.numpy())
class_weights_sampler = 1.0 / class_counts_train
sample_weights = class_weights_sampler[y_train.numpy()]
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float),
    num_samples=len(sample_weights),
    replacement=True
)

train_ds = MelDataset(X_train, y_train, augment=True,  freq_mask=10, time_mask=20)
val_ds   = MelDataset(X_val,   y_val,   augment=False)
test_ds  = MelDataset(X_test,  y_test,  augment=False)

BATCH_SIZE   = 32
USE_MIXUP    = True   # Mettre False pour désactiver
MIXUP_ALPHA  = 0.2

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f"✅ DataLoaders prêts")
print(f"   Batch size : {BATCH_SIZE}")
print(f"   SpecAugment : freq_mask=10, time_mask=20")
print(f"   Mixup : {'activé (alpha=0.2)' if USE_MIXUP else 'désactivé'}")

## 6. Architecture CNNBiLSTMAttention — entrée 2 canaux

> **Seul changement vs v1** : `Conv2d(2, 32, ...)` au lieu de `Conv2d(1, 32, ...)`.
> Tout le reste (BatchNorm, LSTM, Attention, FC) est identique.

In [ ]:
class CNNBiLSTMAttention(nn.Module):
    """
    CNN (features locaux) → BiLSTM (dynamique temporelle)
    → Self-Attention (instants clés) → Classifieur 5 classes.

    Paramètre in_channels : 1 = Mel seul, 2 = Mel + MFCC (v2).
    L'input_size du LSTM est calculé dynamiquement.
    """

    def __init__(self, num_classes: int, in_channels: int = 2,
                 hidden_size: int = 128, dropout: float = 0.5):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_classes = num_classes
        self.dropout_p   = dropout

        # --- Bloc CNN (in_channels=2 pour Mel+MFCC) ---
        self.cnn = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1),  # ← 2 au lieu de 1
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),   # conserve la résolution temporelle
        )

        # Couches dynamiques (initialisées au premier forward)
        self.lstm = None
        self.attn = None
        self.fc   = None

    def _build_dynamic_layers(self, cnn_out: torch.Tensor):
        """Calcule automatiquement l'input_size du LSTM selon la shape réelle."""
        feat_size = cnn_out.size(1) * cnn_out.size(2)  # C * m'
        print(f"   ✅ lstm_input_size calculé dynamiquement : {feat_size}")

        self.lstm = nn.LSTM(
            input_size=feat_size,
            hidden_size=self.hidden_size,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        ).to(cnn_out.device)

        self.attn = nn.Sequential(
            nn.Linear(self.hidden_size * 2, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        ).to(cnn_out.device)

        self.fc = nn.Sequential(
            nn.Dropout(self.dropout_p),
            nn.Linear(self.hidden_size * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, self.num_classes)
        ).to(cnn_out.device)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b = x.size(0)
        cnn_out = self.cnn(x)                              # (B, 128, m', t')

        if self.lstm is None:
            self._build_dynamic_layers(cnn_out)

        # (B, 128, m', t') → (B, t', 128*m')
        seq = cnn_out.permute(0, 3, 1, 2).reshape(b, cnn_out.size(3), -1)

        lstm_out, _ = self.lstm(seq)                       # (B, t', 2H)
        attn_w  = torch.softmax(self.attn(lstm_out), dim=1)
        context = torch.sum(attn_w * lstm_out, dim=1)     # (B, 2H)
        return self.fc(context)                            # (B, num_classes)


# Instanciation avec in_channels=2
IN_CHANNELS = X.shape[1]   # 2 si Mel+MFCC, 1 si Mel seul
model = CNNBiLSTMAttention(
    num_classes=len(class_mapping),
    in_channels=IN_CHANNELS
).to(DEVICE)

# Dry-run pour déclencher l'init dynamique
with torch.no_grad():
    _ = model(X_train[:2].to(DEVICE))

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"   in_channels = {IN_CHANNELS} ({'Mel+MFCC' if IN_CHANNELS==2 else 'Mel seul'})")
print(f"   Paramètres entraînables : {total_params:,}")

## 7. Entraînement — Warmup + Cosine + Label Smoothing + Mixup

> **Nouveautés v2** :
> - `label_smoothing=0.1` dans la loss → meilleure généralisation sur classes similaires
> - Warmup linéaire 5 epochs → LR monte progressivement, évite les oscillations initiales
> - Mixup en training → frontières plus douces entre Asthma et Bronchial
> - 120 epochs + patience 20 → plus de temps pour converger

In [ ]:
# === Poids de classes pour la loss ===
class_counts_train_np = np.bincount(y_train.numpy())
class_weights_loss = len(y_train) / (len(class_mapping) * class_counts_train_np)
weights_tensor = torch.tensor(class_weights_loss, dtype=torch.float).to(DEVICE)
print("Poids loss par classe :")
for cls, w in zip(sorted(class_mapping.keys()), weights_tensor.cpu().numpy()):
    print(f"  {cls:12s} : {w:.3f}")

# === Loss avec label smoothing (v2) ===
criterion = nn.CrossEntropyLoss(
    weight=weights_tensor,
    label_smoothing=0.1    # ← nouveauté v2
)

# === Optimiseur ===
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

# === Scheduler : warmup 5 epochs puis cosine (v2) ===
EPOCHS   = 120
PATIENCE = 20
WARMUP   = 5

warmup_scheduler = optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.1, end_factor=1.0, total_iters=WARMUP
)
cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS - WARMUP, eta_min=1e-6
)
scheduler = optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[WARMUP]
)

print(f"\nConfig entraînement :")
print(f"  Epochs max   : {EPOCHS}")
print(f"  Early stop   : patience {PATIENCE}")
print(f"  Warmup       : {WARMUP} epochs")
print(f"  Label smooth : 0.1")
print(f"  Mixup        : {'activé' if USE_MIXUP else 'désactivé'}")

In [ ]:
def evaluate(loader):
    """Évalue le modèle sur un DataLoader. Retourne loss, acc, f1, preds, labels, probs."""
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    total_loss = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            out   = model(xb)
            loss  = criterion(out, yb)
            total_loss += loss.item()
            probs = torch.softmax(out, dim=1)
            all_preds.extend(out.argmax(1).cpu().numpy())
            all_labels.extend(yb.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    avg_loss = total_loss / len(loader)
    acc  = accuracy_score(all_labels, all_preds)
    f1   = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, acc, f1, np.array(all_preds), np.array(all_labels), np.array(all_probs)


# =====================================================================
# Boucle d'entraînement
# =====================================================================
best_val_f1 = 0.0
best_state  = None
no_improve  = 0
history     = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "val_f1": []}

print(f"\n🚀 Démarrage de l'entraînement...")
print(f"{'Epoch':>6} | {'TrainLoss':>9} | {'ValLoss':>8} | {'TrainAcc':>9} | {'ValAcc':>7} | {'ValF1':>7} | {'LR':>9}")
print("-" * 80)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_total = 0
    train_preds, train_labels = [], []

    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()

        # --- Mixup (v2) ---
        if USE_MIXUP and np.random.random() > 0.5:
            xb_mix, ya, yb_mix, lam = mixup_batch(xb, yb, alpha=MIXUP_ALPHA)
            out  = model(xb_mix)
            loss = lam * criterion(out, ya) + (1 - lam) * criterion(out, yb_mix)
        else:
            out  = model(xb)
            loss = criterion(out, yb)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss_total += loss.item()
        train_preds.extend(out.argmax(1).cpu().numpy())
        train_labels.extend(yb.cpu().numpy())

    scheduler.step()

    train_loss = train_loss_total / len(train_loader)
    train_acc  = accuracy_score(train_labels, train_preds)
    val_loss, val_acc, val_f1, _, _, _ = evaluate(val_loader)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)

    current_lr = optimizer.param_groups[0]['lr']
    print(f"{epoch:>6} | {train_loss:>9.4f} | {val_loss:>8.4f} | "
          f"{train_acc:>9.3f} | {val_acc:>7.3f} | {val_f1:>7.3f} | {current_lr:.3e}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state  = {k: v.clone() for k, v in model.state_dict().items()}
        no_improve  = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"\n⏹️  Early stopping epoch {epoch} | Meilleur Val F1 = {best_val_f1:.4f}")
            break

print(f"\n✅ Entraînement terminé | Meilleur Val Macro F1 = {best_val_f1:.4f}")
model.load_state_dict(best_state)

## 8. Courbes d'apprentissage

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
epochs_ran = range(1, len(history["train_loss"]) + 1)

axes[0].plot(epochs_ran, history["train_loss"], label="Train", color="steelblue")
axes[0].plot(epochs_ran, history["val_loss"],   label="Val",   color="tomato")
axes[0].axvline(x=WARMUP, color="gray", linestyle=":", alpha=0.6, label="Fin warmup")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_ran, history["train_acc"], label="Train", color="steelblue")
axes[1].plot(epochs_ran, history["val_acc"],   label="Val",   color="tomato")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].plot(epochs_ran, history["val_f1"], color="seagreen", label="Val Macro F1")
axes[2].axhline(0.90, color="gray", linestyle="--", alpha=0.6, label="Benchmark 0.90")
axes[2].set_title("Macro F1 (Val)")
axes[2].set_xlabel("Epoch")
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.suptitle("CNNBiLSTMAttention v2 — Mel + MFCC", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("/tmp/learning_curves_v2.png", dpi=120, bbox_inches="tight")
plt.show()

## 9. Évaluation finale — Test set + TTA

> **TTA (Test Time Augmentation)** : on prédit 5 versions légèrement augmentées
> de chaque sample et on moyenne les probabilités. Réduit la variance sans rechanger le modèle.

In [ ]:
def evaluate_tta(X_data, y_data, n_passes=5):
    """
    Évaluation avec Test Time Augmentation.
    Pour chaque sample, on moyenne les probabilités sur n_passes augmentations légères.
    """
    model.eval()
    all_probs_mean = []

    # Dataset augmenté léger pour TTA (masques réduits)
    tta_ds = MelDataset(X_data, y_data, augment=True, freq_mask=8, time_mask=15)

    with torch.no_grad():
        for i in range(len(X_data)):
            probs_list = []
            for _ in range(n_passes):
                xb, _ = tta_ds[i]
                xb = xb.unsqueeze(0).to(DEVICE)
                prob = torch.softmax(model(xb), dim=1).cpu().squeeze()
                probs_list.append(prob)
            all_probs_mean.append(torch.stack(probs_list).mean(0))

    all_probs = torch.stack(all_probs_mean).numpy()
    all_preds = all_probs.argmax(axis=1)
    all_labels = y_data.numpy()

    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    try:
        roc = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")
    except Exception:
        roc = float("nan")
    return acc, f1, roc, all_preds, all_labels, all_probs


# --- Évaluation standard (sans TTA) ---
_, test_acc_std, test_f1_std, preds_std, labels_std, probs_std = evaluate(test_loader)
try:
    roc_std = roc_auc_score(labels_std, probs_std, multi_class="ovr", average="macro")
except Exception:
    roc_std = float("nan")

# --- Évaluation avec TTA ---
print("⏳ TTA en cours (5 passes par sample)...")
test_acc_tta, test_f1_tta, roc_tta, preds_tta, labels_tta, probs_tta = evaluate_tta(
    X_test, y_test, n_passes=5
)

class_names = sorted(class_mapping.keys())

print("\n" + "=" * 60)
print("         🏆 RÉSULTATS TEST SET")
print("=" * 60)
print(f"  {'Métrique':<20} {'Sans TTA':>10} {'Avec TTA':>10} {'Benchmark':>10}")
print("-" * 60)
print(f"  {'Accuracy':<20} {test_acc_std:>10.4f} {test_acc_tta:>10.4f} {'~0.91':>10}")
print(f"  {'Macro F1':<20} {test_f1_std:>10.4f} {test_f1_tta:>10.4f} {'~0.90':>10}")
print(f"  {'Macro ROC-AUC':<20} {roc_std:>10.4f} {roc_tta:>10.4f} {'~0.99':>10}")
print("=" * 60)

print("\n📋 Rapport détaillé (avec TTA) :")
print(classification_report(labels_tta, preds_tta, target_names=class_names))

## 10. Matrice de confusion

In [ ]:
cm      = confusion_matrix(labels_tta, preds_tta)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title("Matrice de confusion (counts) — TTA")
axes[0].set_xlabel("Prédit")
axes[0].set_ylabel("Réel")

sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title("Matrice de confusion (normalisée) — TTA")
axes[1].set_xlabel("Prédit")
axes[1].set_ylabel("Réel")

plt.suptitle("CNNBiLSTMAttention v2 — Mel + MFCC", fontsize=12)
plt.tight_layout()
plt.savefig("/tmp/confusion_matrix_v2.png", dpi=120, bbox_inches="tight")
plt.show()

## 11. Sauvegarde du modèle dans Snowflake Stage

In [ ]:
MODEL_STAGE = "@M2_ISD_EQUIPE_1_DB.PROCESSED.STG_RESPIRATORY_FEATURES"
MODEL_PATH  = "/tmp/cnn_bilstm_v2.pt"
META_PATH   = "/tmp/model_metadata_v2.pkl"

torch.save({
    "model_state_dict" : model.state_dict(),
    "class_mapping"    : class_mapping,
    "in_channels"      : IN_CHANNELS,
    "n_mels"           : X.shape[2],
    "time_frames"      : X.shape[3],
    "hidden_size"      : 128,
    "num_classes"      : len(class_mapping),
    "version"          : "v2_mel_mfcc",
    "test_acc_tta"     : test_acc_tta,
    "test_f1_tta"      : test_f1_tta,
    "test_roc_tta"     : roc_tta,
}, MODEL_PATH)

meta = {
    "class_mapping" : class_mapping,
    "in_channels"   : int(IN_CHANNELS),
    "n_mels"        : int(X.shape[2]),
    "time_frames"   : int(X.shape[3]),
    "version"       : "v2_mel_mfcc",
}
with open(META_PATH, "wb") as f:
    pickle.dump(meta, f)

session.file.put(MODEL_PATH, f"{MODEL_STAGE}/models", auto_compress=False, overwrite=True)
session.file.put(META_PATH,  f"{MODEL_STAGE}/models", auto_compress=False, overwrite=True)

print(f"✅ Modèle v2 sauvegardé dans {MODEL_STAGE}/models/")
print(f"   Accuracy TTA = {test_acc_tta:.4f}")
print(f"   F1 TTA       = {test_f1_tta:.4f}")
print(f"   ROC-AUC TTA  = {roc_tta:.4f}")

## 12. Fonction d'inférence Streamlit — avec TTA

In [ ]:
def load_model_from_stage(stage_path: str = f"{MODEL_STAGE}/models"):
    """
    Charge le modèle v2 depuis Snowflake Stage.
    Utilisé par l'app Streamlit en production.
    """
    tmp = tempfile.mkdtemp()
    session.file.get(f"{stage_path}/cnn_bilstm_v2.pt", tmp)
    ckpt = torch.load(os.path.join(tmp, "cnn_bilstm_v2.pt"), map_location="cpu")

    m = CNNBiLSTMAttention(
        num_classes=ckpt["num_classes"],
        in_channels=ckpt["in_channels"],
        hidden_size=ckpt["hidden_size"]
    )
    dummy = torch.zeros(1, ckpt["in_channels"], ckpt["n_mels"], ckpt["time_frames"])
    with torch.no_grad():
        _ = m(dummy)
    m.load_state_dict(ckpt["model_state_dict"])
    m.eval()
    return m, ckpt["class_mapping"]


def predict_mel_mfcc(mel: np.ndarray, mfcc: np.ndarray,
                     model_instance, class_map: dict,
                     use_tta: bool = True, n_passes: int = 5) -> dict:
    """
    Prédit la classe à partir d'un Mel + MFCC.
    use_tta=True : moyenne sur n_passes augmentations (recommandé).
    Retourne un dict {classe: probabilité} trié par score décroissant.
    """
    def preprocess(m, mc):
        m_norm  = (m  - m.mean())  / (m.std()  + 1e-8)
        mc_norm = (mc - mc.mean()) / (mc.std() + 1e-8)
        mc_r = F.interpolate(
            torch.tensor(mc_norm, dtype=torch.float32).unsqueeze(0).unsqueeze(0),
            size=(m_norm.shape[0], m_norm.shape[1]),
            mode="bilinear", align_corners=False
        ).squeeze(0).squeeze(0)
        return torch.stack([torch.tensor(m_norm, dtype=torch.float32), mc_r], dim=0)

    tensor = preprocess(mel, mfcc)

    if use_tta:
        aug_ds = MelDataset(
            tensor.unsqueeze(0),
            torch.zeros(1, dtype=torch.long),
            augment=True, freq_mask=8, time_mask=15
        )
        probs_list = []
        with torch.no_grad():
            for _ in range(n_passes):
                xb, _ = aug_ds[0]
                p = torch.softmax(model_instance(xb.unsqueeze(0)), dim=1).squeeze()
                probs_list.append(p)
        probs = torch.stack(probs_list).mean(0).numpy()
    else:
        with torch.no_grad():
            probs = torch.softmax(
                model_instance(tensor.unsqueeze(0)), dim=1
            ).squeeze().numpy()

    idx_to_class = {v: k for k, v in class_map.items()}
    return {idx_to_class[i]: float(probs[i]) for i in np.argsort(probs)[::-1]}


# Test rapide
example_mel  = X_test[0][0].numpy()   # canal 0 = Mel
example_mfcc = X_test[0][1].numpy()   # canal 1 = MFCC
predictions  = predict_mel_mfcc(example_mel, example_mfcc, model, class_mapping, use_tta=True)

print("🔬 Exemple de prédiction (avec TTA) :")
for cls, prob in predictions.items():
    bar = "█" * int(prob * 30)
    print(f"  {cls:12s} : {prob:.2%}  {bar}")